In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_PATH = Path('/content/drive/MyDrive/AI/GMAv3/best_models')

In [ ]:
model_names = list(map(lambda x: x.name, BASE_PATH.iterdir()))

In [ ]:
from transformers import AutoConfig, AutoModel, AutoTokenizer
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'roberta-base'

class GMAv3(nn.Module):
    def __init__(self, num_classes=6):
        super(GMAv3, self).__init__()
        self.huggingface_config = AutoConfig.from_pretrained(model_name)
        self.huggingface = AutoModel.from_config(self.huggingface_config)
        self.fc = nn.Linear(self.huggingface_config.hidden_size, num_classes)

    def forward(self, x, att_masks):
        x = self.huggingface(x, att_masks)
        x = x.last_hidden_state[:, 0, :]
        x = self.fc(x)
        return x


def load_model(model_name):
    state_dict = torch.load(BASE_PATH / model_name, map_location=DEVICE, weights_only=True)['model_state_dict']
    model = GMAv3()
    model.load_state_dict(state_dict)
    return model

In [ ]:
import json
author_dict = {}
with open(BASE_PATH / '../author_dict.json', 'r') as f:
    author_dict = json.load(f)

authors = list(author_dict.values())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
models = list(map(load_model, model_names))

In [ ]:
from tabulate import tabulate
from termcolor import colored

message = input("Message: ")
ids = tokenizer.encode(message, padding=True, truncation=False, return_tensors='pt')
att_masks = torch.ones_like(ids)

preds = []
for model in models:
    pred = model(ids, att_masks)[0]
    pred = F.softmax(pred, dim=0)
    preds.append(pred)

preds = np.array([t.detach().numpy() for t in preds]).T
avgs = preds.mean(axis=1)
preds = np.concatenate((preds, avgs.reshape(-1, 1)), axis=1)
preds = preds.round(3)
preds = preds.tolist()

headers = [f'model_{i}' for i in range(1, 5)] + ['average']
authors_censored = [f"{s[0]}{(len(s) - 2) * '*'}{s[-1]}" for s in authors]
row_names = authors_censored

print(tabulate(preds, headers=headers, tablefmt='grid', showindex=row_names))